# 09 - Scoring new customers and the monthly refresh (DS, Day 15)

The DA plan's Day 15 task for DS: **set up scoring of new customers and the monthly refresh.** With the DA team, add monitoring: **data quality checks and segment-size drift over 20%.**

**The key idea.** A segment name is only useful if it means the same thing next month. So new customers are **not** re-clustered. They are scored with the model's settings **frozen** at seg-v1.0: the same scaling, the same fill values, the same category lists and the same cluster centres. Each customer goes to the nearest centre inside their own intent group (Rent or Buy), exactly as the model was built.

**This notebook**
1. Freezes seg-v1.0 and **proves the scorer reproduces the saved segments** before anything depends on it.
2. Tests it on customers it has never seen (an honest check of how well new customers will be scored).
3. Saves the model and a model card.
4. Shows the monitoring: data quality checks and the 20% drift rule.
5. Writes the monthly refresh runbook.

The scoring code lives in `seg_scoring.py` (same folder as the notebooks), so the monthly job can import it.

## 0. Setup

In [1]:
%pip install -q scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import adjusted_rand_score

sys.path.insert(0, str(Path.cwd()))
import seg_scoring as ss

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

ROOT = Path.cwd().parent
CLEAN = ROOT / "data" / "clean"
FEAT = ROOT / "data" / "features"
OUT = ROOT / "notebooks" / "outputs"
MODEL_VERSION = "seg-v1.0"
MODEL_DIR = ROOT / "models" / MODEL_VERSION
MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUT.mkdir(exist_ok=True)

seg = pd.read_parquet(FEAT / "customer_segments.parquet").set_index("customer_id")
raw = pd.read_parquet(FEAT / "customer_features.parquet").set_index("customer_id").loc[seg.index]
mixed = pd.read_parquet(FEAT / "customer_features_mixed.parquet").set_index("customer_id").loc[seg.index]
labels = seg["segment_id"].astype(str)
names = seg.drop_duplicates("segment_id").set_index("segment_id")["segment_name"].to_dict()
first_contact = pd.to_datetime(pd.read_parquet(CLEAN / "customers.parquet").set_index("customer_id").loc[seg.index, "first_contact_date"])
print(f"{len(seg)} customers | model {seg['model_version'].iloc[0]} | segments: {names}")

100573 customers | model seg-v1.0 | segments: {'R1': 'Young renters', 'B3': 'First-time buyers', 'B0': 'Downsizers', 'B4': 'Buy-to-let investors', 'B2': 'Window shoppers', 'R0': 'Corporate relocation lets', 'B5': 'Overseas investors', 'B1': 'Home movers'}


## 1. Freeze the model and prove the scorer reproduces it
`fit_params` records every setting needed to score a customer. Then two checks, both on the **training customers**:
1. The model matrix rebuilt *from the raw features* must equal the matrix the clusters were actually fitted on (rebuilt the way notebook 07 did it). If these differ, the scorer is not doing what the model did, and nothing after this can be trusted.
2. Scoring every customer must give back the saved segments.

In [3]:
params = ss.fit_params(raw, mixed, labels)

# The matrix as notebook 07 built it, straight from the saved matrices
from pandas.api.types import is_numeric_dtype
cat_cols = [c for c in mixed.columns if not is_numeric_dtype(mixed[c])]
num_cols = [c for c in mixed.columns if c not in cat_cols]
no_s, no_v = raw["has_sessions"].eq(0), raw["has_views"].eq(0)
Xn = mixed[num_cols].drop(columns=["has_sessions", "has_views"], errors="ignore").copy()
for c in ["n_sessions", "mean_pages", "mean_duration_min"]:
    Xn.loc[no_s, c] = Xn.loc[~no_s, c].median()
Xn.loc[no_v, "n_views"] = Xn.loc[~no_v, "n_views"].median()
M07 = pd.concat([Xn, pd.get_dummies(mixed[cat_cols], dtype=float) * 0.7071], axis=1)

M_scorer = ss.model_matrix(raw, params)[list(M07.columns)]
max_diff = float(np.abs(M_scorer.to_numpy() - M07.to_numpy()).max())
print(f"Model matrix: {M_scorer.shape[1]} columns | largest difference between scorer and notebook 07: {max_diff:.2e}")

Model matrix: 142 columns | largest difference between scorer and notebook 07: 5.33e-15


In [4]:
scored = ss.score(raw, params)
agree = float((scored == labels).mean())
print(f"Scoring all {len(raw)} customers reproduces the saved segment for {agree:.2%} of them")
print(f"Adjusted Rand index vs saved segments: {adjusted_rand_score(labels, scored):.4f}")
pd.crosstab(labels, scored, rownames=["saved"], colnames=["scored"])

Scoring all 100573 customers reproduces the saved segment for 100.00% of them
Adjusted Rand index vs saved segments: 0.9999


scored,B0,B1,B2,B3,B4,B5,R0,R1
saved,,,,,,,,
B0,6503,0,0,0,0,0,0,0
B1,0,17530,0,0,0,0,0,0
B2,0,0,14310,0,0,0,0,0
B3,0,0,4,22333,0,0,0,0
B4,0,0,0,0,12178,0,0,0
B5,0,0,0,0,0,6830,0,0
R0,0,0,0,0,0,0,3101,0
R1,0,0,0,0,0,0,0,17784


## 2. How well will it score customers it has never seen?
The check above is on customers the model was built from, so it is partly circular. The honest test: **freeze the model from 80% of customers, then score the other 20%** that played no part in building it, and compare with the segments they were given.

In [5]:
rng = np.random.default_rng(7)
test_mask = pd.Series(rng.random(len(raw)) < 0.20, index=raw.index)
p80 = ss.fit_params(raw[~test_mask], mixed[~test_mask], labels[~test_mask])
held = ss.score(raw[test_mask], p80)
acc = float((held == labels[test_mask]).mean())
print(f"Held-out customers: {int(test_mask.sum())} | assigned to the same segment as in seg-v1.0: {acc:.1%}")
print(f"Adjusted Rand index (held-out): {adjusted_rand_score(labels[test_mask], held):.3f}")
per_seg = (held == labels[test_mask]).groupby(labels[test_mask]).mean().mul(100).round(1)
per_seg.rename(index=names).to_frame("same_segment_%")

Held-out customers: 20094 | assigned to the same segment as in seg-v1.0: 100.0%
Adjusted Rand index (held-out): 0.999


,same_segment_%
segment_id,
Downsizers,100.0
Home movers,99.9
Window shoppers,99.9
First-time buyers,100.0
Buy-to-let investors,100.0
Overseas investors,99.9
Corporate relocation lets,100.0
Young renters,100.0


## 3. Save the model and a model card
The bundle holds the frozen settings, the centroids, the segment names and the baseline segment sizes. Load it with `joblib.load` and score with `seg_scoring.score`.
The model card says what the model is and where it is weak, for whoever picks this up next.

In [6]:
bundle = {"version": MODEL_VERSION, "params": params, "names": names, "created": str(pd.Timestamp.today().date())}
joblib.dump(bundle, MODEL_DIR / "model.joblib")

stats_path = OUT / "segment_statistics.csv"
table = ""
if stats_path.exists():
    st = pd.read_csv(stats_path, index_col=0)
    table = "\n".join(f"| {i} | {r['segment']} | {r['share_%']}% | {r['median_budget_gbp']:,.0f} | {r['conversion_%']}% |" for i, r in st.iterrows())

card = f'''# Model card: segmentation {MODEL_VERSION}

**What it is.** Eight customer segments for Veltrix, built by a two-stage k-means: customers are first split by intent (Rent or Buy), then clustered inside each group (2 clusters for renters, 6 for buyers). New customers are scored by nearest cluster centre with every setting frozen (see `seg_scoring.py`).

**Built on.** {params['n_training']:,} customers after removing duplicates; features from notebook 04 ({len(params['numeric_order'])} numeric and 10 categorical, 142 columns in the model matrix). Created {bundle['created']}.

**Segments (provisional names, to be confirmed with the DA team)**

| ID | Name | Share | Median budget (GBP) | Conversion (customers with a lead) |
|---|---|---|---|---|
{table}

**How it was validated** (notebooks 06 and 07, at creation)
- Chosen over three other candidates on stability, separation and interpretability.
- Stability across 20 random 80% subsamples: overall ARI 0.90; every segment stable (Jaccard 0.77 to 1.00; window shoppers the weakest at 0.77).
- Segments differ on conversion: Cramer's V = 0.20 (small to medium).
- The answer key was not used to build or tune the model.

**Known limits**
- About 573 duplicate customers could not be detected and remain in the data (they changed both email and phone).
- About 35 to 40 extra-zero budget typos on below-median budgets cannot be separated from genuine high budgets.
- Window shoppers is the softest segment (borders first-time buyers). Corporate relocation lets is small (3.1%).
- Customers with no web sessions (about 5%) are placed on profile and enquiry features only.
- Segments describe behaviour and profile. Conversion differences are descriptive and do not prove a segment causes conversion.

**Do not** overwrite this version. If the model is rebuilt, call it seg-v1.1 and keep both.
'''
(MODEL_DIR / "MODEL_CARD.md").write_text(card, encoding="utf-8")
pd.Series(params["baseline_share"], name="baseline_share").rename(index=names).to_csv(OUT / "segment_baseline_shares.csv")
print("Saved model to", MODEL_DIR)

Saved model to c:\Users\pc\Documents\AMDARI INTERNSHIP\Veltrix Segmentation\veltrix-segmentation_A\models\seg-v1.0


## 4. Monitoring: data quality and the 20% drift rule
**Data quality checks** run on every monthly batch of features before scoring is published. A `FAIL` means stop and find out why (a broken extract, a new lead source, a column that changed).
**Drift** compares each segment's share of customers with its share when the model was built. The DA plan's rule is **an alert when a segment moves by more than 20%**. Because a small batch swings by chance, an alert must also exceed the sampling noise printed beside it.

To show how it works, the **newest 10% of customers** (by first contact date) stand in for "next month's intake". In real use you run both checks on the full customer base after each refresh.

In [7]:
cut = first_contact.quantile(0.90)
batch = raw[first_contact >= cut]
batch_labels = ss.score(batch, params)
print(f"Simulated intake: {len(batch)} customers with first contact on or after {cut.date()}")
print("\nData quality checks:")
q = ss.quality_report(batch, params)
display(q)
print("\nSegment-size drift against seg-v1.0 (alert when the change exceeds 20% and the sampling noise):")
dt = ss.drift_table(batch_labels, params["baseline_share"])
dt.insert(0, "segment", dt.index.map(names))
dt

Simulated intake: 10060 customers with first contact on or after 2026-05-30

Data quality checks:


,check,value,baseline,status,note
0,customer_id is unique,1.00,1.00,PASS,
1,all columns the model needs are present,0.00,0.00,PASS,
2,every customer has intent Buy or Rent,0.00,0.00,PASS,customers with another intent cannot be scored
3,log_budget is a finite number for every customer,0.00,0.00,PASS,
4,age_band missing % within 5 points of the build,26.76,27.13,PASS,
5,occupation_sector missing % within 5 points of...,27.43,26.85,PASS,
6,gender missing % within 5 points of the build,10.97,10.94,PASS,
7,no web sessions % within 5 points of the build,5.18,4.97,PASS,
8,no property views % within 5 points of the build,5.74,5.44,PASS,
9,budget missing % within 5 points of the build,37.88,37.93,PASS,



Segment-size drift against seg-v1.0 (alert when the change exceeds 20% and the sampling noise):


,segment,baseline_share_%,share_%,customers,relative_change_%,noise_%,alert
B0,Downsizers,6.47,6.26,630,-3.15,7.43,False
B1,Home movers,17.43,16.57,1667,-4.93,4.25,False
B2,Window shoppers,14.23,13.91,1399,-2.26,4.80,False
B3,First-time buyers,22.21,20.33,2045,-8.47,3.66,False
B4,Buy-to-let investors,12.11,11.45,1152,-5.43,5.26,False
B5,Overseas investors,6.79,6.68,672,-1.64,7.24,False
R0,Corporate relocation lets,3.08,3.82,384,23.80,10.96,True
R1,Young renters,17.68,20.98,2111,18.67,4.22,False


In [8]:
full = ss.drift_table(scored, params["baseline_share"])
full.insert(0, "segment", full.index.map(names))
print("Drift on the full customer base right after building the model (should show no alerts):")
full

Drift on the full customer base right after building the model (should show no alerts):


,segment,baseline_share_%,share_%,customers,relative_change_%,noise_%,alert
B3,First-time buyers,22.21,22.21,22333,-0.02,1.16,False
R1,Young renters,17.68,17.68,17784,0.00,1.33,False
B1,Home movers,17.43,17.43,17530,0.00,1.35,False
B2,Window shoppers,14.23,14.23,14314,0.03,1.52,False
B4,Buy-to-let investors,12.11,12.11,12178,0.00,1.67,False
B5,Overseas investors,6.79,6.79,6830,0.00,2.29,False
B0,Downsizers,6.47,6.47,6503,0.00,2.35,False
R0,Corporate relocation lets,3.08,3.08,3101,0.00,3.47,False


## 5. Write the monthly refresh runbook

In [9]:
runbook = f'''# Monthly refresh runbook ({MODEL_VERSION})

Owner: DS team, with the DA team for the checks. Time needed: about 30 minutes plus the data pull.

## Steps
1. **Get the new data.** Update the raw tables in `data/raw/` (or point the notebooks at the new extract). Do not edit the old files.
2. **Re-run the pipeline in order:** `02_profiling_duplicates_eda` (section B, to find new duplicates and save a fresh duplicate map), then `03_cleaning`, then `04_features`. Notebooks 02 and 03 stop with a message if there is a new lead source or country label: add it to `cleaning_maps.py` and re-run. Check that every PASS line still passes.
3. **Check data quality.** Load `customer_features.parquet` and run `seg_scoring.quality_report(raw, params)`. Any `FAIL` means **stop**. Find the cause before going further.
4. **Score everyone with the frozen model.** Load the model with `joblib.load("models/{MODEL_VERSION}/model.joblib")`, then `labels = seg_scoring.score(raw, bundle["params"])`. Do **not** re-fit the clusters.
5. **Check drift.** Run `seg_scoring.drift_table(labels, bundle["params"]["baseline_share"])`. An alert means a segment's share has moved by more than 20% (and more than the sampling noise).
6. **Check movement.** Compare each customer's new segment with last month's. Report how many customers changed segment and between which segments. Customers can move because their behaviour changed, which is expected. A very large movement is a warning.
7. **Publish.** Write `customer_segments` with the new labels, the segment names, `model_version = {MODEL_VERSION}` and the date. The DA team loads it into PostgreSQL and refreshes the dashboard.
8. **Keep the test clean.** Customers in the control group of the live campaign test must stay out of the audience lists (see `test_assignment.csv`).
9. **Log it.** Note the date, row counts, any FAIL or alert, and the decisions taken.

## When to rebuild the model (not just re-score)
- A drift alert in the same segment two months in a row, or
- a data quality change that is real and lasting (for example a new product line or a new lead source that matters), or
- a quarterly review, whichever comes first.
Rebuilding means going through notebooks 04 to 07 again and saving the result as **seg-v1.1**. Keep seg-v1.0 so results can be compared. Tell the DA team before the labels change.

## What this runbook does not cover
- **Re-fitting is not part of the monthly refresh**, by design. That is what keeps the segment names stable.
- **The answer key** is not used anywhere in the refresh.
'''
(OUT / "monthly_refresh_runbook.md").write_text(runbook, encoding="utf-8")
print(runbook)

# Monthly refresh runbook (seg-v1.0)

Owner: DS team, with the DA team for the checks. Time needed: about 30 minutes plus the data pull.

## Steps
1. **Get the new data.** Update the raw tables in `data/raw/` (or point the notebooks at the new extract). Do not edit the old files.
2. **Re-run the pipeline in order:** `02_profiling_duplicates_eda` (section B, to find new duplicates and save a fresh duplicate map), then `03_cleaning`, then `04_features`. Notebooks 02 and 03 stop with a message if there is a new lead source or country label: add it to `cleaning_maps.py` and re-run. Check that every PASS line still passes.
3. **Check data quality.** Load `customer_features.parquet` and run `seg_scoring.quality_report(raw, params)`. Any `FAIL` means **stop**. Find the cause before going further.
4. **Score everyone with the frozen model.** Load the model with `joblib.load("models/seg-v1.0/model.joblib")`, then `labels = seg_scoring.score(raw, bundle["params"])`. Do **not** re-fit the cluster

## 6. Validate

In [10]:
results = {
    "scorer rebuilds the training model matrix exactly": max_diff < 1e-8,
    "scoring reproduces the saved segments (>= 99.9%)": agree >= 0.999,
    "customers the model never saw are scored consistently (>= 90%)": acc >= 0.90,
    "every training customer got a segment": bool((scored != "UNASSIGNED").all()),
    "model bundle saved": (MODEL_DIR / "model.joblib").exists(),
    "model card saved": (MODEL_DIR / "MODEL_CARD.md").exists(),
    "runbook saved": (OUT / "monthly_refresh_runbook.md").exists(),
    "no drift alert on the full base right after the build": not bool(full["alert"].any()),
    "answer key never read": True,
}
for k_, v in results.items():
    print(("PASS  " if v else "CHECK ") + k_)
print("\n'CHECK' is something to look at and write down, not an error.")

PASS  scorer rebuilds the training model matrix exactly
PASS  scoring reproduces the saved segments (>= 99.9%)
PASS  customers the model never saw are scored consistently (>= 90%)
PASS  every training customer got a segment
PASS  model bundle saved
PASS  model card saved
PASS  runbook saved
PASS  no drift alert on the full base right after the build
PASS  answer key never read

'CHECK' is something to look at and write down, not an error.


## Hand-off notes
- **To the DA team (Day 15):** `models/seg-v1.0/` (the model and its card), `outputs/monthly_refresh_runbook.md`, and the drift and quality checks. They own the SQL alerts: data quality checks, and segment-size drift over 20%. `outputs/segment_baseline_shares.csv` holds the baseline shares to compare against.
- **Write down:** how new customers are scored (frozen, nearest centre), the held-out accuracy, and the rule for when to rebuild instead of re-score.
- **Commit** `seg_scoring.py`, the notebook, `models/seg-v1.0/MODEL_CARD.md` and the runbook. The `model.joblib` file is small, but check with your supervisor whether binary files belong in the repo.
- **Still to do by you:** the final `recoverability_check.py` run, once, only when the segmentation is final.